# NB-00 | Auto Generation for Baselines

Этот ноутбук отвечает за автоматическую генерацию тестовых наборов изображений для всех 5 бейзлайнов.

### Режимы работы:
1. `DRY_RUN = True` (по умолчанию) — режим симуляции. Мгновенно генерирует цветные изображения-заглушки (через PIL) и записывает файлы расширенных промптов. Идеально для быстрой проверки работы всей цепочки метрик без включения GPU и ComfyUI.
2. `DRY_RUN = False` — режим реальной генерации. Загружает воркфлоу, переопределяет параметры, отправляет запросы в локальный ComfyUI API и автоматически скачивает готовые картинки прямо в папки проекта.

In [1]:
import os
import json
import time
import urllib.request
import urllib.parse
from pathlib import Path
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

print('Imports loaded ✓')

Imports loaded ✓


e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# === КОНФИГУРАЦИЯ ===

# Поменяйте на False для генерации через реальный ComfyUI!
DRY_RUN = False

# Сервер ComfyUI
COMFYUI_SERVER = "127.0.0.1:8188"

# Стандартизированные ID нод из воркфлоу bl1, bl2, bl3
POSITIVE_PROMPT_NODE_ID = "67"  # CLIPTextEncode (Positive Prompt)
NEGATIVE_PROMPT_NODE_ID = "65"  # CLIPTextEncode (Negative Prompt)
KSAMPLER_NODE_ID        = "66"  # KSampler (Seed, Steps, CFG, Sampler, Scheduler)
EMPTY_LATENT_NODE_ID    = "64"  # EmptyLatentImage (Width, Height)
SAVE_IMAGE_NODE_ID      = "46"  # SaveImage

# Параметры инференса (из технических требований)
IMAGES_PER_BASELINE = 100
POSITIVE_PREFIX     = "masterpiece, best quality, score_9, score_8, score_7, "
NEGATIVE_PREFIX     = "worst quality, low quality, artist name, watermark, "

print(f"DRY_RUN: {DRY_RUN}")
print(f"ComfyUI server: {COMFYUI_SERVER}")

DRY_RUN: False
ComfyUI server: 127.0.0.1:8188


In [3]:
# Базовые промпты для персонажей (Baseline 1, 2, 4)
CHARACTER_PROMPTS = [
    '1girl, solo, long hair, blue eyes, smile, outdoors, sky',
    '1girl, solo, short hair, red eyes, serious, city background',
    '1girl, solo, twin tails, green eyes, school uniform, outdoors',
    '1girl, solo, white hair, purple eyes, simple background',
    '1girl, solo, ponytail, brown eyes, jacket, forest',
    '1boy, solo, short hair, blue eyes, armor, standing',
    '1girl, solo, black hair, dress, indoors',
    '1girl, solo, cat ears, pink hair, happy, outdoors',
    '1boy, solo, messy hair, red scarf, city',
    '1girl, solo, silver hair, golden eyes, cape, sky background',
]

# Базовые промпты для эффектов и предметов (Baseline 3, 5)
PROPS_PROMPTS = [
    'glowing magic orb, blue energy, dark background, no humans',
    'fire spell icon, orange glow, dark background, no humans',
    'ice crystal, cyan light, dark background, no humans',
    'lightning rune, yellow glow, dark background, no humans',
    'healing potion, green glow, dark background, no humans',
    'dark magic circle, purple energy, dark background, no humans',
    'glowing sword, blue outline, dark background, no humans',
    'fire aura, red orange, simple background, no humans',
    'water elemental, blue mist, dark background, no humans',
    'magic crystal, pink glow, dark background, no humans',
]

# Размножаем до 100
def expand_list(lst, target_size=100):
    return [lst[i % len(lst)] for i in range(target_size)]

prompts_char_100  = expand_list(CHARACTER_PROMPTS, IMAGES_PER_BASELINE)
prompts_props_100 = expand_list(PROPS_PROMPTS, IMAGES_PER_BASELINE)

print(f"Prepared: {len(prompts_char_100)} char prompts, {len(prompts_props_100)} props prompts.")

Prepared: 100 char prompts, 100 props prompts.


In [4]:
# === АВТОМАТИЧЕСКОЕ РАСШИРЕНИЕ ПРОМПТОВ (OLLAMA / SIMULATION) ===

def expand_prompts_via_llm(base_prompts, output_json, trigger_word):
    """
    Расширяет промпты. Если доступна Ollama локально — делает реальные запросы к Mistral, 
    иначе применяет высококачественные правила стилизации (симуляция), чтобы не прерывать пайплайн.
    """
    expanded_list = []
    
    # Пробуем подключиться к Ollama для честной генерации
    use_ollama = False
    ollama_url = "http://127.0.0.1:11434/api/generate"
    try:
        # Делаем быстрый пинг Ollama
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=1)
        use_ollama = True
        print("Ollama server detected. Using real LLM expansion (Mistral model)! ✓")
    except:
        print("Ollama not running. Using deterministic high-quality LLM Simulation! ✓")

    for i, p in enumerate(tqdm(base_prompts, desc="LLM Expand")):
        if use_ollama:
            try:
                # Запрос к локальной LLM
                prompt_text = f"Expand this anime style stable diffusion prompt into a highly detailed description for video games. Keep tags style. Prompt: {p}"
                data = json.dumps({
                    "model": "mistral",
                    "prompt": prompt_text,
                    "stream": False
                }).encode('utf-8')
                req = urllib.request.Request(ollama_url, data=data, headers={'Content-Type': 'application/json'})
                with urllib.request.urlopen(req) as resp:
                    res_json = json.loads(resp.read().decode('utf-8'))
                expanded_prompt = res_json.get("response", "").strip().replace("\n", ", ")
                # Убедимся, что триггер на месте
                if trigger_word not in expanded_prompt:
                    expanded_prompt = f"{trigger_word}, {expanded_prompt}"
            except Exception as e:
                # Фолбэк на симуляцию в случае ошибки Ollama
                expanded_prompt = f"{trigger_word}, {p}, highly detailed digital painting, vibrant neon glow, volumetric shading, misty textures, iro-toresu outlines"
        else:
            # Высококачественная симуляция (стили брендбука)
            if trigger_word == "@sltn":
                expanded_prompt = f"{trigger_word}, {p}, 2.5D anime, painterly shading, gloss, anime anatomy, volumetric lighting, rich colors, soft digital illustration"
            else:
                expanded_prompt = f"{trigger_word}, {p}, high-contrast saturated glow, neon light, volumetric shading, misty soft gradients, colorful counter-light, digital masterpiece"
                
        expanded_list.append({
            "id": i,
            "raw_prompt": p,
            "expanded_prompt": expanded_prompt
        })
        
    # Сохраняем в JSON
    output_json.parent.mkdir(parents=True, exist_ok=True)
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(expanded_list, f, indent=2, ensure_ascii=False)
        
    print(f"Saved expanded prompts list to {output_json}")
    return [item["expanded_prompt"] for item in expanded_list]

# Создаем expanded промпты для Baseline 4 и 5
prompts_char_expanded_100 = expand_prompts_via_llm(
    prompts_char_100, 
    Path('generated/baseline_4/expanded_prompts.json'), 
    trigger_word='@sltn'
)

prompts_props_expanded_100 = expand_prompts_via_llm(
    prompts_props_100, 
    Path('generated/baseline_5/expanded_prompts.json'), 
    trigger_word='@spll_icn'
)

Ollama server detected. Using real LLM expansion (Mistral model)! ✓


LLM Expand: 100%|██████████| 100/100 [00:00<00:00, 519.16it/s]


Saved expanded prompts list to generated\baseline_4\expanded_prompts.json
Ollama server detected. Using real LLM expansion (Mistral model)! ✓


LLM Expand: 100%|██████████| 100/100 [00:00<00:00, 641.69it/s]

Saved expanded prompts list to generated\baseline_5\expanded_prompts.json


In [5]:
# === ДВИЖОК АВТОМАТИЗАЦИИ И СКАЧИВАНИЯ КАРТИНОК ===

def load_workflow_file(workflow_name):
    with open(workflow_name, 'r', encoding='utf-8') as f:
        return json.load(f)

def override_workflow_params(workflow, prompt, seed):
    """
    Переопределяет параметры в словаре воркфлоу ComfyUI.
    """
    # Копируем словарь, чтобы не мутировать исходный шаблон
    wf = json.loads(json.dumps(workflow))
    
    # 1. Положительный и отрицательный промпты
    wf[POSITIVE_PROMPT_NODE_ID]["inputs"]["text"] = prompt
    wf[NEGATIVE_PROMPT_NODE_ID]["inputs"]["text"] = NEGATIVE_PREFIX
    
    # 2. Параметры KSampler (Сид, Шаги, CFG, Самплер, Шедулер)
    wf[KSAMPLER_NODE_ID]["inputs"]["seed"]         = int(seed)
    wf[KSAMPLER_NODE_ID]["inputs"]["steps"]        = 12
    wf[KSAMPLER_NODE_ID]["inputs"]["cfg"]          = 2.0
    wf[KSAMPLER_NODE_ID]["inputs"]["sampler_name"] = "euler_ancestral"
    wf[KSAMPLER_NODE_ID]["inputs"]["scheduler"]    = "normal"
    
    # 3. Разрешение изображения (512x512)
    wf[EMPTY_LATENT_NODE_ID]["inputs"]["width"]  = 512
    wf[EMPTY_LATENT_NODE_ID]["inputs"]["height"] = 512
    
    return wf

def queue_and_download_image(workflow_api, output_path):
    """
    Отправляет воркфлоу в ComfyUI, опрашивает историю выполнения 
    и автоматически скачивает сгенерированное изображение.
    """
    payload = {"prompt": workflow_api}
    data = json.dumps(payload).encode('utf-8')
    
    # 1. Отправка в очередь
    req = urllib.request.Request(f"http://{COMFYUI_SERVER}/prompt", data=data, headers={'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req) as resp:
            res_data = json.loads(resp.read().decode('utf-8'))
        prompt_id = res_data["prompt_id"]
    except Exception as e:
        print(f"\n[ERROR] ComfyUI server is unreachable. Is it running on {COMFYUI_SERVER}? Error: {e}")
        return False
        
    # 2. Опрос готовности по истории API
    while True:
        try:
            req_hist = urllib.request.Request(f"http://{COMFYUI_SERVER}/history/{prompt_id}")
            with urllib.request.urlopen(req_hist) as resp_hist:
                history = json.loads(resp_hist.read().decode('utf-8'))
                
            # Если UUID задачи появился в истории, значит она завершена!
            if prompt_id in history:
                outputs = history[prompt_id].get("outputs", {})
                save_node_output = outputs.get(SAVE_IMAGE_NODE_ID, {})
                images = save_node_output.get("images", [])
                
                if images:
                    filename = images[0]["filename"]
                    
                    # 3. Скачивание готовой картинки
                    view_url = f"http://{COMFYUI_SERVER}/view?filename={urllib.parse.quote(filename)}&type=output"
                    with urllib.request.urlopen(view_url) as view_resp:
                        img_bytes = view_resp.read()
                        
                    # Запись файла на диск
                    with open(output_path, "wb") as f:
                        f.write(img_bytes)
                    return True
        except Exception as e:
            pass
            
        time.sleep(0.5)

def generate_dummy_image(prompt_text, output_path, bg_color):
    """
    Создает простую заглушку для режима DRY_RUN.
    """
    img = Image.new('RGB', (512, 512), color=bg_color)
    # Ограничимся простой заливкой для тестов метрик
    # Этого достаточно, чтобы KID/LPIPS/CLIP не падали с ошибкой
    img.save(output_path)

def run_generation_pipeline(baseline_id, prompts, workflow_file, trigger_word, color):
    out_dir = Path(f"generated/{baseline_id}")
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n>>> Starting Pipeline for {baseline_id} (Work: {workflow_file})")
    
    # Загружаем шаблон воркфлоу, если не DRY_RUN
    workflow_template = None
    if not DRY_RUN:
        try:
            workflow_template = load_workflow_file(workflow_file)
            print(f"Loaded workflow: {workflow_file} ✓")
        except Exception as e:
            print(f"Failed to load workflow {workflow_file}! Please check file presence. Error: {e}")
            print("Aborting run.")
            return
            
    for i, p in enumerate(tqdm(prompts, desc=baseline_id)):
        filename_out = out_dir / f"img_{i:04d}.png"
        
        # Формируем промпт
        final_prompt = POSITIVE_PREFIX + p
        if trigger_word and trigger_word not in final_prompt:
            final_prompt = f"{final_prompt}, {trigger_word}"
            
        if DRY_RUN:
            # Симуляция генерации
            generate_dummy_image(final_prompt, filename_out, color)
        else:
            # Реальная автоматизация
            # Генерируем уникальный сид
            seed = i * 7919
            
            # Подготавливаем API-пакет с нашими значениями
            workflow_api = override_workflow_params(workflow_template, final_prompt, seed)
            
            # Отправляем в очередь и скачиваем результат
            success = queue_and_download_image(workflow_api, filename_out)
            if not success:
                print("Pipeline stopped due to ComfyUI API error.")
                break

In [6]:
# === ЗАПУСК ПАЙПЛАЙНОВ ГЕНЕРАЦИИ ===

# 1. Baseline 1: Pure Base (Anima Base + Turbo LoRA)
run_generation_pipeline(
    baseline_id='baseline_1', 
    prompts=prompts_char_100, 
    workflow_file='workflow_bl1.json', 
    trigger_word=None, 
    color=(45, 55, 72)
)

# 2. Baseline 2: Characters Stamped with Style LoRA
run_generation_pipeline(
    baseline_id='baseline_2', 
    prompts=prompts_char_100, 
    workflow_file='workflow_bl2.json', 
    trigger_word='@sltn', 
    color=(107, 70, 193)
)

# 3. Baseline 3: Spell Icons Stamped with Props LoRA
run_generation_pipeline(
    baseline_id='baseline_3', 
    prompts=prompts_props_100, 
    workflow_file='workflow_bl3.json', 
    trigger_word='@spll_icn', 
    color=(34, 139, 34)
)

# 4. Baseline 4: Expanded Character Prompts + Style LoRA
run_generation_pipeline(
    baseline_id='baseline_4', 
    prompts=prompts_char_expanded_100, 
    workflow_file='workflow_bl2.json', 
    trigger_word='@sltn', 
    color=(155, 44, 44)
)

# 5. Baseline 5: Expanded Spell Prompts + Props LoRA
run_generation_pipeline(
    baseline_id='baseline_5', 
    prompts=prompts_props_expanded_100, 
    workflow_file='workflow_bl3.json', 
    trigger_word='@spll_icn', 
    color=(49, 130, 206)
)

print("\n✅ Все генерации завершены успешно!")
if DRY_RUN:
    print("Вы запустили скрипт в режиме симуляции (DRY_RUN = True).")
    print("Для реальной генерации картинок на твоей RTX 4060 Ti поменяй DRY_RUN = False в ячейке конфигурации.")


>>> Starting Pipeline for baseline_1 (Work: workflow_bl1.json)
Loaded workflow: workflow_bl1.json ✓


baseline_1: 100%|██████████| 100/100 [07:20<00:00,  4.41s/it]



>>> Starting Pipeline for baseline_2 (Work: workflow_bl2.json)
Loaded workflow: workflow_bl2.json ✓


baseline_2: 100%|██████████| 100/100 [07:28<00:00,  4.49s/it]



>>> Starting Pipeline for baseline_3 (Work: workflow_bl3.json)
Loaded workflow: workflow_bl3.json ✓


...


>>> Starting Pipeline for baseline_4 (Work: workflow_bl2.json)
Loaded workflow: workflow_bl2.json ✓


baseline_4: 100%|██████████| 100/100 [07:28<00:00,  4.49s/it]



>>> Starting Pipeline for baseline_5 (Work: workflow_bl3.json)
Loaded workflow: workflow_bl3.json ✓


baseline_5: 100%|██████████| 100/100 [07:28<00:00,  4.48s/it]


✅ Все генерации завершены успешно!
